# Monthly Synthesis Diagnostics: Snow DA, Soil Moisture, and Flux Pathways

Exploratory monthly-resolution diagnostics for the M21C land-sweeper manuscript.

This notebook tests whether MODIS snow-cover assimilation leaves a seasonal soil-moisture imprint, whether that imprint projects onto model fluxes when flux variables are available, and whether snow DA activity changes later microwave soil-moisture DA "work".

Important constraint: these diagnostics use monthly files only. They support language such as seasonal propagation, spring-to-summer carryover, and monthly/seasonal association. They do not support precise melt-out or 0-30 day post-melt timing claims.

Primary outputs are written to `projects/M21C_ls/output/monthly_synthesis_diagnostics/`.


## Diagnostic Plan

1. **Inventory:** confirm monthly files, variables, units, and available diagnostics.
2. **Analysis 1:** MODIS-only period, MAM snow DA activity vs AMJ/MJJ/JJA soil-moisture response.
3. **Analysis 2:** ET/evaporative-fraction response, if monthly flux variables are available.
4. **Analysis 3:** MAM snow DA activity vs later monthly soil-moisture DA work using monthly analysis increments.

Positive signed state differences are `DA - OL`. Positive absolute activity/work metrics are magnitudes. These are exploratory DA-impact diagnostics, not independent validation metrics.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import xarray as xr

try:
    from IPython import get_ipython
    from IPython.display import display
except Exception:
    def get_ipython():
        return None
    def display(obj):
        print(obj)


def running_in_notebook() -> bool:
    shell = get_ipython()
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"


IN_NOTEBOOK = running_in_notebook()

import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as exc:
    HAS_CARTOPY = False
    print("Cartopy unavailable; map cells will fall back to lon/lat scatter:", exc)


def show_figure(fig):
    if IN_NOTEBOOK:
        display(fig)
    plt.close(fig)


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / ".git").exists() and (p / "common/python/io/read_GEOSldas.py").exists():
            return p
    raise FileNotFoundError("Could not locate geosldas-analysis repo root")


HERE = Path.cwd().resolve()
REPO_ROOT = find_repo_root(HERE)
PROJECT_ROOT = REPO_ROOT / "projects/M21C_ls"
OUT_DIR = PROJECT_ROOT / "output/monthly_synthesis_diagnostics"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2")
INPUTS = {
    "ol_land_monthly": DATA_DIR / "OLv8_land_variables_2000_2024_compressed.nc",
    "da_land_monthly": DATA_DIR / "DAv8_land_variables_2000_2024_compressed.nc",
    "monthly_increments": DATA_DIR / "LS_monthly_increments_2000_2024.nc",
    "da_monthly_ofa_pickle": DATA_DIR / "spatial_stats_DA_200006_202405.pkl",
    "ol_monthly_ofa_pickle": DATA_DIR / "spatial_stats_OL_200006_202405.pkl",
}
AUX_MONTHLY_INPUTS = {
    "flux_core": {
        "ol": DATA_DIR / "OLv8_flux_core_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_flux_core_2000_2024_compressed.nc",
    },
    "latent_components": {
        "ol": DATA_DIR / "OLv8_latent_components_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_latent_components_2000_2024_compressed.nc",
    },
    "water_budget": {
        "ol": DATA_DIR / "OLv8_water_budget_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_water_budget_2000_2024_compressed.nc",
    },
    "energy_context": {
        "ol": DATA_DIR / "OLv8_energy_context_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_energy_context_2000_2024_compressed.nc",
    },
}

PERIODS = {
    "modis_only": {"label": "MODIS-only / snow-only DA period", "start": "2000-06-01", "end": "2007-05-31"},
    "pre_smap_mw": {"label": "microwave pre-SMAP period", "start": "2007-06-01", "end": "2015-03-31"},
    "smap_era": {"label": "SMAP-era microwave period", "start": "2015-04-01", "end": "2024-05-31"},
}

SEASON_WINDOWS = {
    "MAM": [3, 4, 5],
    "AMJ": [4, 5, 6],
    "MJJ": [5, 6, 7],
    "JJA": [6, 7, 8],
}

# Complete calendar years used for seasonal monthly windows. 2007 is excluded
# from Analysis 1 response windows because ASCAT starts in June 2007.
ANALYSIS1_YEARS = list(range(2001, 2007))
ANALYSIS3_PRE_SMAP_YEARS = list(range(2008, 2015))
ANALYSIS3_SMAP_YEARS = list(range(2016, 2024))

SNOW_POSSIBLE_SCF_THRESHOLD = 0.05
PERMANENT_SNOW_JJA_MEAN_MAX = 0.20
NH_MIN_LAT = 20.0
N_BINS = 8

print("REPO_ROOT:", REPO_ROOT)
print("OUT_DIR:", OUT_DIR)
print("IN_NOTEBOOK:", IN_NOTEBOOK)


In [ ]:
def describe_dataset(path: Path) -> pd.DataFrame:
    rows = []
    if not path.exists():
        return pd.DataFrame([{"path": str(path), "exists": False}])
    if path.suffix not in {".nc", ".nc4"}:
        return pd.DataFrame([{
            "path": str(path),
            "exists": True,
            "file": path.name,
            "variable": "file",
            "dims": "",
            "shape": "",
            "dtype": path.suffix.lstrip("."),
            "units": "",
            "long_name": "non-NetCDF auxiliary input",
        }])
    with xr.open_dataset(path) as ds:
        for name, da in ds.data_vars.items():
            rows.append({
                "path": str(path),
                "exists": True,
                "file": path.name,
                "variable": name,
                "dims": ",".join(da.dims),
                "shape": "x".join(str(da.sizes[d]) for d in da.dims),
                "dtype": str(da.dtype),
                "units": da.attrs.get("units", ""),
                "long_name": da.attrs.get("long_name", ""),
            })
        for name in ds.coords:
            if name not in ds.data_vars:
                da = ds[name]
                rows.append({
                    "path": str(path),
                    "exists": True,
                    "file": path.name,
                    "variable": f"coord:{name}",
                    "dims": ",".join(da.dims),
                    "shape": "x".join(str(da.sizes[d]) for d in da.dims),
                    "dtype": str(da.dtype),
                    "units": da.attrs.get("units", ""),
                    "long_name": da.attrs.get("long_name", ""),
                })
    return pd.DataFrame(rows)

aux_input_paths = [path for pair in AUX_MONTHLY_INPUTS.values() for path in pair.values()]
all_input_paths = list(INPUTS.values()) + aux_input_paths
inventory = pd.concat([describe_dataset(path) for path in all_input_paths], ignore_index=True)
inventory.to_csv(OUT_DIR / "monthly_synthesis_input_inventory.csv", index=False)
display(inventory)

ol_land_vars = set(inventory.loc[inventory.file == INPUTS["ol_land_monthly"].name, "variable"])
da_land_vars = set(inventory.loc[inventory.file == INPUTS["da_land_monthly"].name, "variable"])
aux_var_sources = {}
ol_aux_vars = set()
da_aux_vars = set()
for group_name, paths in AUX_MONTHLY_INPUTS.items():
    group_ol_vars = set(inventory.loc[inventory.file == paths["ol"].name, "variable"])
    group_da_vars = set(inventory.loc[inventory.file == paths["da"].name, "variable"])
    ol_aux_vars |= group_ol_vars
    da_aux_vars |= group_da_vars
    for var_name in group_ol_vars & group_da_vars:
        aux_var_sources[var_name] = group_name
ol_available_vars = ol_land_vars | ol_aux_vars
da_available_vars = da_land_vars | da_aux_vars
land_vars = ol_land_vars
increment_vars = set(inventory.loc[inventory.file == INPUTS["monthly_increments"].name, "variable"])

flux_candidates = [
    "EVLAND", "LHLAND", "SHLAND",
    "LHLANDINTR", "LHLANDSBLN", "LHLANDSOIL", "LHLANDTRNS",
    "GHLAND", "SWLAND", "LWLAND",
    "SMLAND", "RUNSURFLAND", "BASEFLOWLAND", "QINFILLAND",
    "TWLAND", "WCHANGELAND",
]
flux_available = sorted(v for v in flux_candidates if v in ol_available_vars and v in da_available_vars)
state_required = ["SFMC", "RZMC", "FRLANDSNO", "SNOMASLAND", "SNODPLAND", "TSOIL1"]
missing_state = sorted(v for v in state_required if v not in land_vars)
increment_available = sorted(v for v in ["SFMC_INC", "RZMC_INC", "SNOWMASS_INCR"] if v in increment_vars)

availability_summary = pd.DataFrame([
    {"category": "state_variables_required", "status": "missing" if missing_state else "available", "variables": ", ".join(missing_state) if missing_state else ", ".join(state_required)},
    {"category": "flux_variables", "status": "available" if flux_available else "not available locally", "variables": ", ".join(flux_available)},
    {"category": "monthly_increments", "status": "available" if increment_available else "not available locally", "variables": ", ".join(increment_available)},
])
availability_summary.to_csv(OUT_DIR / "monthly_synthesis_availability_summary.csv", index=False)
display(availability_summary)


## Helper Functions

These helpers keep the calculations explicit and monthly/seasonal. The seasonal windows are calendar-year windows only; this first pass does not attempt melt-out timing.


In [ ]:
def open_ds(path: Path, vars_keep=None):
    if not path.exists():
        raise FileNotFoundError(path)
    ds = xr.open_dataset(path, decode_times=True)
    if vars_keep is None:
        return ds
    keep = [v for v in vars_keep if v in ds.data_vars]
    coord_keep = [c for c in ["lat", "lon", "tile"] if c in ds.coords]
    return ds[keep + coord_keep]


def month_selector(da, year: int, months: list[int]):
    return (da.time.dt.year == year) & da.time.dt.month.isin(months)


def seasonal_mean(da, years: list[int], months: list[int], strict: bool = True, name: str | None = None):
    pieces = []
    valid_years = []
    for year in years:
        sub = da.sel(time=month_selector(da, year, months))
        if strict and sub.sizes.get("time", 0) != len(months):
            warnings.warn(f"Skipping {year}: expected {len(months)} months, found {sub.sizes.get('time', 0)}")
            continue
        if sub.sizes.get("time", 0) == 0:
            continue
        pieces.append(sub.mean("time", skipna=True))
        valid_years.append(year)
    if not pieces:
        raise ValueError(f"No valid seasonal means for years={years}, months={months}")
    out = xr.concat(pieces, dim=pd.Index(valid_years, name="year"))
    if name:
        out.name = name
    return out


def savefig(fig, stem: str):
    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"
    fig.savefig(png, dpi=180, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("saved", png)
    print("saved", pdf)


def add_geo_base(ax, extent=None):
    if not HAS_CARTOPY:
        ax.grid(True, alpha=0.25)
        return
    ax.coastlines(linewidth=0.5, color="0.35")
    ax.add_feature(cfeature.LAND, facecolor="0.92", edgecolor="none", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="white", edgecolor="none", zorder=0)
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())


def tile_scatter_map(ax, lon, lat, values, mask=None, title="", cmap="RdBu_r", norm=None, s=1.0, extent=None):
    lon = np.asarray(lon)
    lat = np.asarray(lat)
    values = np.asarray(values)
    valid = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(values)
    if mask is not None:
        valid &= np.asarray(mask)
    if HAS_CARTOPY:
        add_geo_base(ax, extent=extent)
        im = ax.scatter(lon[valid], lat[valid], c=values[valid], s=s, marker="s", cmap=cmap, norm=norm,
                        linewidths=0, transform=ccrs.PlateCarree(), rasterized=True)
    else:
        add_geo_base(ax, extent=extent)
        im = ax.scatter(lon[valid], lat[valid], c=values[valid], s=s, marker="s", cmap=cmap, norm=norm, linewidths=0)
        if extent is not None:
            ax.set_xlim(extent[0], extent[1])
            ax.set_ylim(extent[2], extent[3])
    ax.set_title(title, fontsize=10)
    return im


def symmetric_norm(values, percentile=98, floor=1e-8):
    arr = np.asarray(values)
    vmax = np.nanpercentile(np.abs(arr[np.isfinite(arr)]), percentile) if np.isfinite(arr).any() else floor
    vmax = max(float(vmax), floor)
    return mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)


def binned_summary(df, x_col, y_col, by_cols=None, n_bins=8):
    by_cols = by_cols or []
    frames = []
    group_iter = [((), df)] if not by_cols else df.groupby(by_cols, dropna=False)
    for key, sub in group_iter:
        sub = sub[[x_col, y_col] + by_cols].replace([np.inf, -np.inf], np.nan).dropna(subset=[x_col, y_col])
        if sub.empty or sub[x_col].nunique() < 2:
            continue
        try:
            bins = pd.qcut(sub[x_col], q=n_bins, duplicates="drop")
        except ValueError:
            continue
        grouped = sub.assign(bin=bins).groupby("bin", observed=True)
        out = grouped.agg(
            n=(y_col, "size"),
            x_mean=(x_col, "mean"),
            x_min=(x_col, "min"),
            x_max=(x_col, "max"),
            y_mean=(y_col, "mean"),
            y_median=(y_col, "median"),
            y_std=(y_col, "std"),
        ).reset_index(drop=True)
        out["y_se"] = out["y_std"] / np.sqrt(out["n"])
        if by_cols:
            if not isinstance(key, tuple):
                key = (key,)
            for col, val in zip(by_cols, key):
                out[col] = val
        out["x_metric"] = x_col
        out["y_metric"] = y_col
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def make_table_from_yearly_arrays(years, lat, lon, mask, arrays: dict[str, xr.DataArray]) -> pd.DataFrame:
    rows = []
    tile = np.arange(lat.size)
    base_mask = np.asarray(mask, dtype=bool)
    for year in years:
        rec = {
            "year": np.full(base_mask.sum(), year, dtype=int),
            "tile": tile[base_mask],
            "lat": np.asarray(lat)[base_mask],
            "lon": np.asarray(lon)[base_mask],
        }
        for name, arr in arrays.items():
            rec[name] = np.asarray(arr.sel(year=year).values)[base_mask]
        rows.append(pd.DataFrame(rec))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


## Open Monthly Datasets and Build Seasonal-Snow Mask

The first-pass seasonal-snow mask is intentionally simple and Northern Hemisphere focused:

- latitude > 20 N;
- maximum OL/DA monthly snow-cover fraction > 0.05;
- mean JJA snow-cover fraction < 0.20 to avoid permanent snow/glacier-like cells.


In [ ]:
state_keep = ["SFMC", "RZMC", "FRLANDSNO", "SNOMASLAND", "SNODPLAND", "TSOIL1", "PRECTOTCORRLAND"]
ol = open_ds(INPUTS["ol_land_monthly"], state_keep)
da = open_ds(INPUTS["da_land_monthly"], state_keep)

flux_in_state = [v for v in flux_available if v in ol.data_vars and v in da.data_vars]
flux_from_aux = [v for v in flux_available if v not in flux_in_state]
for group_name in sorted({aux_var_sources[v] for v in flux_from_aux}):
    group_vars = [v for v in flux_from_aux if aux_var_sources.get(v) == group_name]
    paths = AUX_MONTHLY_INPUTS[group_name]
    ol_aux = open_ds(paths["ol"], group_vars)
    da_aux = open_ds(paths["da"], group_vars)
    if not np.array_equal(ol.time.values, ol_aux.time.values) or not np.array_equal(da.time.values, da_aux.time.values):
        raise ValueError(f"Land-state and {group_name} monthly time coordinates differ")
    if ol.sizes["tile"] != ol_aux.sizes["tile"] or da.sizes["tile"] != da_aux.sizes["tile"]:
        raise ValueError(f"Land-state and {group_name} tile dimensions differ")
    ol = xr.merge([ol, ol_aux[group_vars]], compat="override")
    da = xr.merge([da, da_aux[group_vars]], compat="override")

land_keep = state_keep + flux_available
inc = open_ds(INPUTS["monthly_increments"], increment_available)

if not np.array_equal(ol.time.values, da.time.values):
    raise ValueError("OL and DA monthly time coordinates differ")
if ol.sizes["tile"] != da.sizes["tile"]:
    raise ValueError("OL and DA tile dimensions differ")

lat = ol["lat"].load()
lon = ol["lon"].load()

delta = xr.Dataset({v: da[v] - ol[v] for v in land_keep if v in da and v in ol})

scf_pair_max = xr.concat([ol["FRLANDSNO"], da["FRLANDSNO"]], dim="experiment").max("experiment")
scf_any = scf_pair_max.max("time", skipna=True).load()
jja_scf = scf_pair_max.sel(time=scf_pair_max.time.dt.month.isin([6, 7, 8])).mean("time", skipna=True).load()
seasonal_snow_mask = ((lat > NH_MIN_LAT) & (scf_any > SNOW_POSSIBLE_SCF_THRESHOLD) & (jja_scf < PERMANENT_SNOW_JJA_MEAN_MAX)).load()
snow_possible_mask = ((lat > NH_MIN_LAT) & (scf_any > SNOW_POSSIBLE_SCF_THRESHOLD)).load()
warm_snowfree_mask = ((np.abs(lat) < 60) & (scf_any < SNOW_POSSIBLE_SCF_THRESHOLD)).load()

mask_summary = pd.DataFrame([
    {"mask": "NH snow possible", "n_tiles": int(snow_possible_mask.sum()), "definition": f"lat>{NH_MIN_LAT} and max monthly OL/DA FRLANDSNO>{SNOW_POSSIBLE_SCF_THRESHOLD}"},
    {"mask": "NH seasonal snow", "n_tiles": int(seasonal_snow_mask.sum()), "definition": f"snow possible and mean JJA FRLANDSNO<{PERMANENT_SNOW_JJA_MEAN_MAX}"},
    {"mask": "warm mostly snow-free", "n_tiles": int(warm_snowfree_mask.sum()), "definition": f"abs(lat)<60 and max monthly OL/DA FRLANDSNO<{SNOW_POSSIBLE_SCF_THRESHOLD}"},
])
mask_summary.to_csv(OUT_DIR / "monthly_synthesis_mask_summary.csv", index=False)
display(mask_summary)


## Analysis 1: MODIS-Only Snow DA Activity and Seasonal Soil-Moisture Response

This is the cleanest causal diagnostic because the selected years avoid microwave soil-moisture assimilation. We use 2001-2006 for MAM predictors and AMJ/MJJ/JJA responses, excluding 2007 response windows because ASCAT begins in June 2007.

Snow DA activity is represented by DA-OL differences in FRLANDSNO and SNOMASLAND. These are proxies for snow assimilation impact, not actual snow analysis increments.


In [ ]:
years1 = ANALYSIS1_YEARS
mam_dscf = seasonal_mean(delta["FRLANDSNO"], years1, SEASON_WINDOWS["MAM"], name="mam_dscf")
mam_abs_dscf = seasonal_mean(abs(delta["FRLANDSNO"]), years1, SEASON_WINDOWS["MAM"], name="mam_abs_dscf")
mam_dswe = seasonal_mean(delta["SNOMASLAND"], years1, SEASON_WINDOWS["MAM"], name="mam_dswe")
mam_abs_dswe = seasonal_mean(abs(delta["SNOMASLAND"]), years1, SEASON_WINDOWS["MAM"], name="mam_abs_dswe")

response_arrays = {
    "sfmc_response_amj": seasonal_mean(delta["SFMC"], years1, SEASON_WINDOWS["AMJ"]),
    "rzmc_response_amj": seasonal_mean(delta["RZMC"], years1, SEASON_WINDOWS["AMJ"]),
    "sfmc_response_mjj": seasonal_mean(delta["SFMC"], years1, SEASON_WINDOWS["MJJ"]),
    "rzmc_response_mjj": seasonal_mean(delta["RZMC"], years1, SEASON_WINDOWS["MJJ"]),
    "sfmc_response_jja": seasonal_mean(delta["SFMC"], years1, SEASON_WINDOWS["JJA"]),
    "rzmc_response_jja": seasonal_mean(delta["RZMC"], years1, SEASON_WINDOWS["JJA"]),
}

analysis1_arrays = {
    "snow_activity_signed_scf_mam": mam_dscf.load(),
    "snow_activity_abs_scf_mam": mam_abs_dscf.load(),
    "snow_activity_signed_swe_mam": mam_dswe.load(),
    "snow_activity_abs_swe_mam": mam_abs_dswe.load(),
}
for key, val in response_arrays.items():
    analysis1_arrays[key] = val.load()

analysis1_table = make_table_from_yearly_arrays(years1, lat.values, lon.values, seasonal_snow_mask.values, analysis1_arrays)
analysis1_table["seasonal_snow_mask"] = True
analysis1_table.to_csv(OUT_DIR / "analysis1_gridcell_year_snow_to_sm_table.csv", index=False)
print(f"Analysis 1 grid-cell-year rows: {len(analysis1_table):,}")
display(analysis1_table.head())

map_ds1 = xr.Dataset(
    {
        "mam_mean_dscf": mam_dscf.mean("year", skipna=True),
        "mam_mean_abs_dscf": mam_abs_dscf.mean("year", skipna=True),
        "mam_mean_dswe": mam_dswe.mean("year", skipna=True),
        "mam_mean_abs_dswe": mam_abs_dswe.mean("year", skipna=True),
        "amj_mean_dsfmc": response_arrays["sfmc_response_amj"].mean("year", skipna=True),
        "amj_mean_drzmc": response_arrays["rzmc_response_amj"].mean("year", skipna=True),
        "mjj_mean_dsfmc": response_arrays["sfmc_response_mjj"].mean("year", skipna=True),
        "mjj_mean_drzmc": response_arrays["rzmc_response_mjj"].mean("year", skipna=True),
        "jja_mean_drzmc": response_arrays["rzmc_response_jja"].mean("year", skipna=True),
        "seasonal_snow_mask": seasonal_snow_mask,
    },
    coords={"tile": np.arange(ol.sizes["tile"]), "lat": lat, "lon": lon},
)
map_ds1.to_netcdf(OUT_DIR / "analysis1_seasonal_snow_to_sm_maps.nc")
print("wrote", OUT_DIR / "analysis1_seasonal_snow_to_sm_maps.nc")


In [ ]:
bin_frames = []
for x_col in ["snow_activity_abs_scf_mam", "snow_activity_abs_swe_mam", "snow_activity_signed_scf_mam", "snow_activity_signed_swe_mam"]:
    for y_col in ["sfmc_response_amj", "rzmc_response_amj", "sfmc_response_mjj", "rzmc_response_mjj", "rzmc_response_jja"]:
        out = binned_summary(analysis1_table, x_col, y_col, n_bins=N_BINS)
        if not out.empty:
            bin_frames.append(out)
analysis1_binned = pd.concat(bin_frames, ignore_index=True) if bin_frames else pd.DataFrame()
analysis1_binned.to_csv(OUT_DIR / "analysis1_binned_snow_activity_vs_sm_response.csv", index=False)
display(analysis1_binned.head(20))


In [ ]:
projection = ccrs.Robinson() if HAS_CARTOPY else None
fig = plt.figure(figsize=(12, 7.2))
map_specs = [
    ("mam_mean_dscf", "MAM DA-OL SCF", "RdBu_r", symmetric_norm(map_ds1["mam_mean_dscf"].where(seasonal_snow_mask).values)),
    ("mam_mean_abs_dscf", "MAM |DA-OL SCF|", "viridis", None),
    ("mam_mean_dswe", "MAM DA-OL SWE", "RdBu_r", symmetric_norm(map_ds1["mam_mean_dswe"].where(seasonal_snow_mask).values)),
    ("amj_mean_drzmc", "AMJ DA-OL RZMC", "RdBu_r", symmetric_norm(map_ds1["amj_mean_drzmc"].where(seasonal_snow_mask).values)),
    ("mjj_mean_drzmc", "MJJ DA-OL RZMC", "RdBu_r", symmetric_norm(map_ds1["mjj_mean_drzmc"].where(seasonal_snow_mask).values)),
    ("jja_mean_drzmc", "JJA DA-OL RZMC", "RdBu_r", symmetric_norm(map_ds1["jja_mean_drzmc"].where(seasonal_snow_mask).values)),
]
for i, (var, title, cmap, norm) in enumerate(map_specs, 1):
    ax = fig.add_subplot(2, 3, i, projection=projection) if HAS_CARTOPY else fig.add_subplot(2, 3, i)
    im = tile_scatter_map(ax, lon.values, lat.values, map_ds1[var].values, mask=seasonal_snow_mask.values,
                          title=title, cmap=cmap, norm=norm, s=0.7, extent=[-180, 180, 20, 90])
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.055, pad=0.04)
    cb.ax.tick_params(labelsize=7)
fig.suptitle("Analysis 1: monthly snow DA activity and seasonal soil-moisture response, 2001-2006", fontsize=13)
savefig(fig, "analysis1_maps_snow_to_sm_response")
show_figure(fig)

plot_df = analysis1_binned[analysis1_binned["x_metric"].eq("snow_activity_abs_scf_mam") & analysis1_binned["y_metric"].isin(["sfmc_response_amj", "rzmc_response_amj", "sfmc_response_mjj", "rzmc_response_mjj"])]
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for metric, sub in plot_df.groupby("y_metric"):
    ax.errorbar(sub["x_mean"], sub["y_mean"], yerr=sub["y_se"], marker="o", linewidth=1.5, capsize=2, label=metric)
ax.axhline(0, color="0.2", linewidth=0.8)
ax.set_xlabel("MAM |DA-OL SCF| bin mean")
ax.set_ylabel("Subsequent DA-OL soil moisture (m3 m-3)")
ax.set_title("Analysis 1: seasonal snow activity vs AMJ/MJJ soil-moisture response")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
savefig(fig, "analysis1_binned_abs_scf_vs_sm_response")
show_figure(fig)


## Analysis 2: ET / Evaporative Fraction Response

This analysis is only possible if monthly flux variables are present in the local monthly land files. The current compressed local files do not contain ET, latent heat, or sensible heat variables, so this section writes an explicit skipped-status table instead of inventing a proxy.


In [ ]:
if flux_available:
    analysis2_status = pd.DataFrame([{
        "analysis": "Analysis 2",
        "status": "flux variables available but detailed ET/EF diagnostics not yet implemented in this exploratory notebook",
        "available_flux_variables": ", ".join(flux_available),
        "caveat": "Confirm units and sign conventions before computing ET or EF.",
    }])
else:
    analysis2_status = pd.DataFrame([{
        "analysis": "Analysis 2",
        "status": "skipped",
        "available_flux_variables": "",
        "caveat": "Local monthly OL/DA compressed land-variable files do not contain ET, LE, or H. Do not infer flux response from state variables alone.",
    }])
analysis2_status.to_csv(OUT_DIR / "analysis2_flux_response_status.csv", index=False)
display(analysis2_status)


## Analysis 3: Snow DA Activity and Later Soil-Moisture DA Work

This section uses monthly analysis increments as the soil-moisture DA work metric. The increment file contains `SFMC_INC` and `RZMC_INC` as monthly ANA-FCST increments. We keep signed and absolute versions separate.

The first pass compares:

- pre-SMAP microwave period years 2008-2014;
- SMAP-era years 2016-2023.


In [ ]:
def build_analysis3_table(period_name, years):
    predictor = {
        "snow_activity_abs_scf_mam": seasonal_mean(abs(delta["FRLANDSNO"]), years, SEASON_WINDOWS["MAM"]).load(),
        "snow_activity_signed_scf_mam": seasonal_mean(delta["FRLANDSNO"], years, SEASON_WINDOWS["MAM"]).load(),
        "snow_activity_abs_swe_mam": seasonal_mean(abs(delta["SNOMASLAND"]), years, SEASON_WINDOWS["MAM"]).load(),
        "snow_activity_signed_swe_mam": seasonal_mean(delta["SNOMASLAND"], years, SEASON_WINDOWS["MAM"]).load(),
    }
    arrays = dict(predictor)
    for season in ["AMJ", "MJJ", "JJA"]:
        for var in ["SFMC_INC", "RZMC_INC"]:
            if var not in inc:
                continue
            signed = seasonal_mean(inc[var], years, SEASON_WINDOWS[season]).load()
            magnitude = seasonal_mean(abs(inc[var]), years, SEASON_WINDOWS[season]).load()
            arrays[f"{var.lower()}_{season.lower()}_signed"] = signed
            arrays[f"{var.lower()}_{season.lower()}_abs"] = magnitude
    table = make_table_from_yearly_arrays(years, lat.values, lon.values, seasonal_snow_mask.values, arrays)
    table["period"] = period_name
    return table

analysis3_tables = []
if {"SFMC_INC", "RZMC_INC"}.issubset(set(inc.data_vars)):
    analysis3_tables.append(build_analysis3_table("pre_smap_mw", ANALYSIS3_PRE_SMAP_YEARS))
    analysis3_tables.append(build_analysis3_table("smap_era", ANALYSIS3_SMAP_YEARS))
    analysis3_table = pd.concat(analysis3_tables, ignore_index=True)
else:
    analysis3_table = pd.DataFrame()

analysis3_table.to_csv(OUT_DIR / "analysis3_gridcell_year_snow_activity_vs_sm_work_table.csv", index=False)
print(f"Analysis 3 grid-cell-year rows: {len(analysis3_table):,}")
display(analysis3_table.head())


In [ ]:
analysis3_binned_frames = []
if not analysis3_table.empty:
    y_metrics = [c for c in analysis3_table.columns if c.endswith("_abs") or c.endswith("_signed")]
    y_metrics = [c for c in y_metrics if c.startswith("sfmc_inc") or c.startswith("rzmc_inc")]
    for x_col in ["snow_activity_abs_scf_mam", "snow_activity_abs_swe_mam", "snow_activity_signed_scf_mam", "snow_activity_signed_swe_mam"]:
        for y_col in y_metrics:
            out = binned_summary(analysis3_table, x_col, y_col, by_cols=["period"], n_bins=N_BINS)
            if not out.empty:
                analysis3_binned_frames.append(out)
analysis3_binned = pd.concat(analysis3_binned_frames, ignore_index=True) if analysis3_binned_frames else pd.DataFrame()
analysis3_binned.to_csv(OUT_DIR / "analysis3_binned_snow_activity_vs_sm_da_work.csv", index=False)
display(analysis3_binned.head(30))

if not analysis3_table.empty:
    smap_years = ANALYSIS3_SMAP_YEARS
    map_ds3 = xr.Dataset(
        {
            "smap_mam_mean_abs_dscf": seasonal_mean(abs(delta["FRLANDSNO"]), smap_years, SEASON_WINDOWS["MAM"]).mean("year", skipna=True),
            "smap_amj_mean_abs_rzmc_inc": seasonal_mean(abs(inc["RZMC_INC"]), smap_years, SEASON_WINDOWS["AMJ"]).mean("year", skipna=True),
            "smap_mjj_mean_abs_rzmc_inc": seasonal_mean(abs(inc["RZMC_INC"]), smap_years, SEASON_WINDOWS["MJJ"]).mean("year", skipna=True),
            "smap_jja_mean_abs_rzmc_inc": seasonal_mean(abs(inc["RZMC_INC"]), smap_years, SEASON_WINDOWS["JJA"]).mean("year", skipna=True),
            "seasonal_snow_mask": seasonal_snow_mask,
        },
        coords={"tile": np.arange(ol.sizes["tile"]), "lat": lat, "lon": lon},
    )
    map_ds3.to_netcdf(OUT_DIR / "analysis3_smapera_snow_activity_and_sm_work_maps.nc")
    print("wrote", OUT_DIR / "analysis3_smapera_snow_activity_and_sm_work_maps.nc")


In [ ]:
if not analysis3_binned.empty:
    plot_df = analysis3_binned[
        analysis3_binned["x_metric"].eq("snow_activity_abs_scf_mam")
        & analysis3_binned["y_metric"].isin(["rzmc_inc_amj_abs", "rzmc_inc_mjj_abs", "rzmc_inc_jja_abs"])
    ]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, period in zip(axes, ["pre_smap_mw", "smap_era"]):
        subp = plot_df[plot_df["period"].eq(period)]
        for metric, sub in subp.groupby("y_metric"):
            ax.errorbar(sub["x_mean"], sub["y_mean"], yerr=sub["y_se"], marker="o", linewidth=1.4, capsize=2, label=metric)
        ax.set_title(period)
        ax.set_xlabel("MAM |DA-OL SCF| bin mean")
        ax.grid(True, alpha=0.25)
    axes[0].set_ylabel("Later |RZMC_INC| (m3 m-3)")
    axes[1].legend(fontsize=8)
    fig.suptitle("Analysis 3: snow DA activity vs later monthly soil-moisture DA work")
    savefig(fig, "analysis3_binned_abs_scf_vs_rzmc_increment_work")
    show_figure(fig)

    projection = ccrs.Robinson() if HAS_CARTOPY else None
    fig = plt.figure(figsize=(11.5, 6.2))
    specs = [
        ("smap_mam_mean_abs_dscf", "SMAP era MAM |DA-OL SCF|"),
        ("smap_amj_mean_abs_rzmc_inc", "SMAP era AMJ |RZMC_INC|"),
        ("smap_mjj_mean_abs_rzmc_inc", "SMAP era MJJ |RZMC_INC|"),
        ("smap_jja_mean_abs_rzmc_inc", "SMAP era JJA |RZMC_INC|"),
    ]
    for i, (var, title) in enumerate(specs, 1):
        ax = fig.add_subplot(2, 2, i, projection=projection) if HAS_CARTOPY else fig.add_subplot(2, 2, i)
        im = tile_scatter_map(ax, lon.values, lat.values, map_ds3[var].values, mask=seasonal_snow_mask.values,
                              title=title, cmap="viridis", norm=None, s=0.8, extent=[-180, 180, 20, 90])
        cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.055, pad=0.04)
        cb.ax.tick_params(labelsize=7)
    fig.suptitle("Analysis 3: SMAP-era snow activity and later RZMC increment magnitude")
    savefig(fig, "analysis3_smapera_maps_snow_activity_and_rzmc_work")
    show_figure(fig)


## Brief Recommendation Scaffold

After running the notebook, use the summary tables and quick-look figures to classify the diagnostics:

- **Analysis 1:** strongest candidate for causal seasonal propagation because only MODIS snow-cover DA is active.
- **Analysis 3:** strongest diagnostic for the snow-SM DA interaction question because it uses actual monthly soil-moisture increments where available.
- **Analysis 2:** currently skipped locally because ET/LE/H flux variables are not in the monthly compressed land-variable files. Revisit if monthly flux files are copied in.

Weak, flat, or noisy relationships are still useful: they would support the interpretation that SCF DA improves snow occurrence more than it changes seasonal hydrologic storage or later microwave SM correction burden.


In [ ]:
recommendation_rows = []
if not analysis1_binned.empty:
    key = analysis1_binned[(analysis1_binned.x_metric == "snow_activity_abs_scf_mam") & (analysis1_binned.y_metric == "rzmc_response_mjj")].copy()
    if not key.empty:
        lo = key.sort_values("x_mean").iloc[0]
        hi = key.sort_values("x_mean").iloc[-1]
        recommendation_rows.append({
            "diagnostic": "Analysis 1",
            "quick_signal": "MJJ RZMC response in highest minus lowest MAM |DA-OL SCF| bin",
            "value": float(hi.y_mean - lo.y_mean),
            "unit": "m3 m-3",
            "interpretation_hint": "Positive means high snow-activity bins are wetter in DA relative to OL than low snow-activity bins.",
        })
if not analysis3_binned.empty:
    key = analysis3_binned[(analysis3_binned.x_metric == "snow_activity_abs_scf_mam") & (analysis3_binned.y_metric == "rzmc_inc_mjj_abs") & (analysis3_binned.period == "smap_era")].copy()
    if not key.empty:
        lo = key.sort_values("x_mean").iloc[0]
        hi = key.sort_values("x_mean").iloc[-1]
        recommendation_rows.append({
            "diagnostic": "Analysis 3",
            "quick_signal": "SMAP-era MJJ |RZMC_INC| in highest minus lowest MAM |DA-OL SCF| bin",
            "value": float(hi.y_mean - lo.y_mean),
            "unit": "m3 m-3",
            "interpretation_hint": "Positive means larger snow-activity bins require larger later RZMC increments; negative means smaller later correction burden.",
        })
recommendation_rows.append({
    "diagnostic": "Analysis 2",
    "quick_signal": "ET/EF response",
    "value": np.nan,
    "unit": "n/a",
    "interpretation_hint": "Skipped until monthly ET/LE/H flux files are available locally.",
})
recommendation = pd.DataFrame(recommendation_rows)
recommendation.to_csv(OUT_DIR / "monthly_synthesis_recommendation_quicklook.csv", index=False)
display(recommendation)
